# 对角化：拆开、分别缩放、合回

继续使用第十五章 A=[[3,1],[1,3]]，不更换应用场景。先手算每一步，再用计算核对；随后改变输入和缩放倍数，观察哪些条件真正重要。

依赖 NumPy、Matplotlib。从新内核按顺序运行；图中文字使用英文。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11})
A = np.array([[3., 1.], [1., 3.]])
v1 = np.array([1., 1.])
v2 = np.array([1., -1.])
P = np.column_stack((v1, v2))
# Derived from c1=(p+q)/2, c2=(p-q)/2, not an unexplained inverse call.
Q = 0.5 * np.array([[1., 1.], [1., -1.]])
D = np.diag([4., 2.])
assert np.allclose(Q @ P, np.eye(2))
assert np.allclose(P @ Q, np.eye(2))
assert np.allclose(A @ v1, 4*v1)
assert np.allclose(A @ v2, 2*v2)


# 1. 同一个向量，两种坐标

x=(3,1) 表示横向3、纵向1。求 c1*v1+c2*v2=x，系数分别是多少？先写出两条方程。

改成 x=(1,0)、(1,1)、(1,-1)，分别预测新坐标。不要把新坐标直接当作原平面的点。


In [ ]:
x = np.array([3., 1.])  # Try [1., 0.], [1., 1.], [1., -1.].
c = Q @ x
print('Original coordinates x:', x)
print('Eigenbasis coefficients c:', c)
print('Rebuild x = P c:', P @ c)
assert np.allclose(P @ c, x)
assert np.allclose(c, [(x[0]+x[1])/2, (x[0]-x[1])/2])


# 2. 分别缩放，再合回

保持刚才的输入。第一份乘4，第二份乘2，新系数是多少？合回原坐标以后呢？

另外预测：若误把 D 直接乘原来的 x，会得到什么？为什么它不是我们要的输出？


In [ ]:
scaled = D @ c
output = P @ scaled
print('Step 1, split:', c)
print('Step 2, scale coefficients:', scaled)
print('Step 3, rebuild:', output)
print('Direct A x:', A @ x)
print('D x without changing coordinates:', D @ x)
assert np.allclose(output, A @ x)
assert np.allclose(A @ P, P @ D)
assert np.allclose(Q @ A @ P, D)
assert np.allclose(P @ D @ Q, A)


# 3. 把两种记录画在各自的坐标轴上

左图的坐标是原来的横纵坐标；右图的坐标是两份特征方向的系数。两幅图记录同一次变换，但坐标轴含义不同。

先预测：右图只需要独立乘4和2；左图是否也能直接独立乘4和2？改变上面的 x 后，从第1节重新运行到这里。


In [ ]:
def draw_record(ax, before, after, xlabel, ylabel, title):
    points = np.vstack((np.zeros(2), before, after))
    low, high = points.min(axis=0), points.max(axis=0)
    span = max(float((high-low).max()), 2.)
    center = (low+high)/2
    ax.set(xlim=(center[0]-.7*span, center[0]+.7*span),
           ylim=(center[1]-.7*span, center[1]+.7*span),
           aspect='equal', xlabel=xlabel, ylabel=ylabel, title=title)
    ax.axhline(0, color='0.65', linewidth=.8)
    ax.axvline(0, color='0.65', linewidth=.8)
    ax.grid(alpha=.18)
    for point, color, label, offset in [(after, '#23836b', 'after', (7, 10)),
                                        (before, '#64717a', 'before', (7, -18))]:
        ax.annotate('', xy=point, xytext=(0, 0),
                    arrowprops={'arrowstyle': '-|>', 'color': color, 'lw': 2.5})
        ax.annotate(f'{label}: ({point[0]:g}, {point[1]:g})', xy=point,
                    xytext=offset, textcoords='offset points', color=color)
fig, axes = plt.subplots(2, 1, figsize=(6.5, 10))
draw_record(axes[0], x, output, 'x1', 'x2', 'Original coordinates: A mixes two entries')
draw_record(axes[1], c, scaled, 'c1', 'c2', 'Eigenbasis coordinates: D scales each entry')
fig.tight_layout()
plt.show()


# 4. 为什么只需拆开一次？

对 x=(3,1)，预测第二次作用后的系数，再合回原坐标。

保持配比记录，连续乘4和2，到最后才合回。与每一步都重新做 A @ x 比较。这里矩阵幂是反复做变换，不是逐元素取幂。


In [ ]:
k = 5  # Try 0, 1, 2, 10.
assert isinstance(k, int) and k >= 0
coefficients_k = np.array([4.**k*c[0], 2.**k*c[1]])
from_coefficients = P @ coefficients_k
repeated = x.copy()
for _ in range(k):
    repeated = A @ repeated
print('Coefficients after k steps:', coefficients_k)
print('Rebuilt output:', from_coefficients)
print('Repeated A output:', repeated)
assert np.allclose(from_coefficients, repeated)
assert np.allclose(np.linalg.matrix_power(A, k), P @ np.linalg.matrix_power(D, k) @ Q)


# 5. 保持方向不变，改两个缩放倍数

如果我们希望第一份乘 alpha、第二份乘 beta，原坐标里的矩阵该怎样改变？先预测，再运行。

依次试 (4,2)、(4,0)、(1,1)、(0.5,-0.5)。观察：零倍数丢掉一个方向，负倍数反向，两个倍数相同则所有向量一起乘同一个数。


In [ ]:
alpha, beta = 4., 0.
D_new = np.diag([alpha, beta])
A_new = P @ D_new @ Q
print('New matrix in original coordinates:\n', A_new)
print('New output:', A_new @ x)
assert np.allclose(A_new @ v1, alpha*v1)
assert np.allclose(A_new @ v2, beta*v2)
assert np.allclose(A_new @ x, P @ (D_new @ (Q @ x)))
if beta == 0:
    assert np.allclose(A_new @ v2, 0)
if alpha == beta:
    assert np.allclose(A_new, alpha*np.eye(2))


# 6. A 不可逆，就不能对角化吗？

上一节取 (alpha,beta)=(4,0)。它仍然有完整的特征基，只是第二份被变成零。

先回答：三步计算用到的是哪个矩阵的逆？A 还是 P？然后检查下面的例子。


In [ ]:
A_zero = P @ np.diag([4., 0.]) @ Q
assert np.linalg.matrix_rank(A_zero) == 1
assert np.linalg.matrix_rank(P) == 2
assert np.allclose(A_zero @ v2, 0)
assert np.allclose(Q @ A_zero @ P, np.diag([4., 0.]))
print('A loses one direction, but P still provides a complete basis.')


# 7. 有无穷多特征向量，为什么还可能不够？

把 v1 与 2*v1 当两列。它们都是特征向量，但只沿同一条直线。

先问：能拼出 v2=(1,-1) 吗？如果要拼出 v1，配比是否唯一？这才是在检查换坐标能不能成立。


In [ ]:
P_bad = np.column_stack((v1, 2*v1))
print('Dependent columns:\n', P_bad)
assert np.linalg.matrix_rank(P_bad) == 1
# The same vector has different coefficient records.
assert np.allclose(P_bad @ np.array([1., 0.]), v1)
assert np.allclose(P_bad @ np.array([-1., 1.]), v1)
# Every generated output has equal entries; v2 does not.
for coefficients in [np.array([1., 0.]), np.array([-1., 1.]), np.array([2., -3.])]:
    generated = P_bad @ coefficients
    assert np.isclose(generated[0], generated[1])
assert not np.isclose(v2[0], v2[1])
print('Cannot describe all inputs, and coefficients are not unique.')


# 8. 同一个重复特征值，两个不同结论

剪切 S=[[1,1],[0,1]] 与单位矩阵 I 都只有特征值1。手解 (S-I)v=0 与 (I-I)v=0。

剪切的特征空间是一条线，单位矩阵的特征空间是整个平面。先判断哪里能选出两支独立向量，再核对维数。


In [ ]:
S = np.array([[1., 1.], [0., 1.]])
I = np.eye(2)
print('S - I:\n', S-I)
print('I - I:\n', I-I)
assert 2-np.linalg.matrix_rank(S-I) == 1
assert 2-np.linalg.matrix_rank(I-I) == 2
assert np.allclose(S @ np.array([1., 0.]), [1., 0.])
assert not np.allclose(S @ np.array([0., 1.]), [0., 1.])
assert np.allclose(I @ np.array([0., 1.]), [0., 1.])


## 合上实验，重建三步

1. 用 x=(5,1) 手算新配比、缩放后的配比和最终输出。
2. 从三个步骤重建 A=PDP^{-1}，再逐列解释 AP=PD。
3. 哪一个逆必须存在？为什么零特征值不阻止对角化？
4. 为什么“特征向量无穷多”不等于“特征方向足够多”？
5. 用剪切与单位矩阵说明，为什么需要求特征空间，而不能只看特征值列表。

参考：[MIT 18.06 第二十二讲](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/29a5c9f572b39efa94c91969f4c1dd8b_MIT18_06S10_L22.pdf)。
